## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

## **Structured Information Extraction with Nugen**
---

### Problem

Businesses produce enormous volumes of unstructured text — emails, incident reports, contracts, support tickets — that contain critical information buried in prose. Extracting fields like dates, amounts, names, or classifications manually is slow, costly, and error-prone.

### Solution

Use Nugen's chat completion API to **extract structured JSON from any unstructured text in a single prompt** — no fine-tuning, no labelled data, no ML expertise required.

### What you will build

A reusable `extract()` function that accepts:
- Any block of unstructured text
- A plain-English description of the fields you want

And returns a validated Python dict ready for a database, API, or downstream pipeline.

### Prerequisites
- A Nugen API key — get one free at [Nugen Dashboard](https://nugen-platform-frontend.azurewebsites.net/dashboard)
- Python 3.9+
- Packages: `requests`, `python-dotenv`

### Step 1 — Install Dependencies

In [ ]:
%pip install requests python-dotenv --quiet

### Step 2 — Import Libraries

In [ ]:
import os
import json
import time
import requests
from dotenv import load_dotenv

### Step 3 — Set Your API Key

Create a `.env` file in the same directory as this notebook:

```bash
NUGEN_API_KEY=your_api_key_here
```

> Never hard-code secrets in a notebook. `python-dotenv` loads the value from `.env` automatically.

### Step 4 — Load Configuration

In [ ]:
load_dotenv()
NUGEN_API_KEY = os.getenv("NUGEN_API_KEY")

if not NUGEN_API_KEY:
    raise EnvironmentError(
        "NUGEN_API_KEY not found. "
        "Create a .env file with NUGEN_API_KEY=your_key "
        "or export the variable in your shell."
    )

NUGEN_API_URL = "https://api.nugen.in/api/v3/inference/chat/completions"
MODEL         = "nugen-flash-instruct"

print("Configuration loaded successfully.")

### Step 5 — The Core `extract()` Function

This is the heart of the cookbook. `extract()` works in three steps:

1. **Prompt construction** — wraps the user's text and field schema into a system prompt that instructs Nugen to respond *only* with valid JSON
2. **API call with retry** — calls Nugen's chat completion endpoint up to 3 times with exponential back-off
3. **JSON parsing and validation** — parses the response and returns a dict; raises a clear error if the model output is malformed

The key design choice is the **system prompt**: by explicitly forbidding any prose outside the JSON block, we make the output reliably machine-parseable across a wide range of input types.

In [ ]:
import re  # needed for JSON extraction

def extract(text: str, schema_description: str) -> dict:
    """Extract structured fields from unstructured text using the Nugen API.

    The model is instructed to respond with a JSON object whose keys match the
    fields described in `schema_description`. If a field is not found, its value
    is null. The parser handles models that wrap JSON in prose or code fences.

    Args:
        text: The unstructured source text (email, report, contract, etc.).
        schema_description: Plain-English list of fields to extract, e.g.:
            "sender_name (string), date (ISO-8601 string), amount_usd (number)"

    Returns:
        A dict of extracted field name -> value pairs.

    Raises:
        RuntimeError: If the API is unreachable after 3 attempts.
        ValueError: If the model response contains no parseable JSON.

    Example:
        >>> result = extract(
        ...     text="Invoice #1042 from Acme Corp. Amount due: $4,250.",
        ...     schema_description="invoice_number (string), vendor (string), amount_usd (number)",
        ... )
        >>> result
        {'invoice_number': '1042', 'vendor': 'Acme Corp.', 'amount_usd': 4250}
    """
    system_prompt = (
        "You are a precise data extraction engine. "
        "Extract the requested fields and return them as a single, valid JSON object. "
        "Use null for any field that cannot be found in the text."
    )

    headers = {
        "Authorization": f"Bearer {NUGEN_API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    f"Fields to extract: {schema_description}\n\n"
                    f"Text:\n{text}"
                ),
            },
        ],
        "temperature": 0.1,
        "max_tokens":  512,
    }

    for attempt in range(3):
        try:
            resp = requests.post(NUGEN_API_URL, json=payload, headers=headers, timeout=30)
            if resp.status_code == 200:
                raw = resp.json()["choices"][0]["message"]["content"].strip()

                # Handle models that wrap JSON in prose or markdown code fences
                fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
                if fence:
                    raw = fence.group(1)
                else:
                    brace = re.search(r"\{.*\}", raw, re.DOTALL)
                    if brace:
                        raw = brace.group(0)

                try:
                    return json.loads(raw)
                except json.JSONDecodeError as e:
                    raise ValueError(f"No JSON found in model response:\n{raw[:300]}") from e

            print(f"  [attempt {attempt+1}] API returned {resp.status_code}: {resp.text[:200]}")
        except requests.RequestException as exc:
            print(f"  [attempt {attempt+1}] Network error: {exc}")
        time.sleep(2 ** attempt)

    raise RuntimeError(
        "Nugen API unreachable after 3 attempts. "
        "Check your NUGEN_API_KEY and network connection."
    )


---

## Examples

The next three sections demonstrate the `extract()` function on progressively complex real-world documents. Each example follows the same pattern:

1. Define the source text
2. Describe the schema in plain English
3. Call `extract()` and pretty-print the result

### Example 1 — Extract Fields from a Support Email

**Input:** A customer support email  
**Goal:** Pull out the sender, issue category, urgency, order number, and a one-line summary

In [ ]:
email_text = """
From: sarah.jones@example.com
Date: 14 March 2025, 09:42
Subject: URGENT - Wrong item received - Order #88412

Hi Support Team,

I placed an order last Tuesday (Order #88412) for a pair of blue running shoes, size 8.
What arrived today was a red yoga mat — completely the wrong item.
I have an important race this weekend and I need the correct shoes by Friday at the latest.
Please arrange an express replacement immediately or I will have to dispute the charge.

Regards,
Sarah Jones
"""

schema = (
    "sender_name (string), "
    "sender_email (string), "
    "order_number (string), "
    "issue_category (one of: wrong_item, missing_item, damaged_item, other), "
    "urgency (one of: low, medium, high, critical), "
    "deadline_mentioned (ISO-8601 date or null), "
    "one_line_summary (string)"
)

result = extract(email_text, schema)
print(json.dumps(result, indent=2))

**Expected output:**

```json
{
  "sender_name": "Sarah Jones",
  "sender_email": "sarah.jones@example.com",
  "order_number": "88412",
  "issue_category": "wrong_item",
  "urgency": "critical",
  "deadline_mentioned": "2025-03-14",
  "one_line_summary": "Customer received wrong item (yoga mat instead of running shoes) and needs express replacement by Friday."
}
```

### Example 2 — Parse an Incident Report

**Input:** An IT security incident report  
**Goal:** Extract structured metadata for an incident management system

In [ ]:
incident_report = """
INCIDENT REPORT — INC-2025-0387
Reported by: Raj Mehta, Senior SRE
Date/Time: 2025-03-10 02:14 UTC

Summary:
The payment processing service (prod-payments-eu) went down at 02:12 UTC due to a
misconfigured load balancer rule deployed in release v4.7.1 at 02:05 UTC.
Approximately 1,200 transactions failed during the 18-minute outage.
The rollback to v4.6.9 was completed at 02:30 UTC.

Root Cause: Human error — incorrect port mapping in the load balancer config.
Severity: P1 (Critical)
Status: Resolved
Follow-up: Post-mortem scheduled for 2025-03-12.
"""

schema = (
    "incident_id (string), "
    "reporter_name (string), "
    "reporter_role (string), "
    "incident_start_utc (ISO-8601 datetime), "
    "resolution_time_utc (ISO-8601 datetime), "
    "affected_service (string), "
    "release_version_at_fault (string), "
    "transactions_impacted (integer or null), "
    "severity (string), "
    "root_cause_category (one of: human_error, software_bug, infrastructure, third_party, unknown), "
    "status (string), "
    "postmortem_date (ISO-8601 date or null)"
)

result = extract(incident_report, schema)
print(json.dumps(result, indent=2))

**Expected output:**

```json
{
  "incident_id": "INC-2025-0387",
  "reporter_name": "Raj Mehta",
  "reporter_role": "Senior SRE",
  "incident_start_utc": "2025-03-10T02:12:00Z",
  "resolution_time_utc": "2025-03-10T02:30:00Z",
  "affected_service": "prod-payments-eu",
  "release_version_at_fault": "v4.7.1",
  "transactions_impacted": 1200,
  "severity": "P1",
  "root_cause_category": "human_error",
  "status": "Resolved",
  "postmortem_date": "2025-03-12"
}
```

### Example 3 — Parse a Purchase Order

**Input:** A free-form purchase order email  
**Goal:** Extract line items as a structured list — demonstrating that the schema can include arrays

In [ ]:
purchase_order = """
Purchase Order from GreenLeaf Supplies
PO Number: PO-9921
Date: 20 March 2025
Ship to: 45 Innovation Drive, Bengaluru 560001

Please supply the following:
- 50 units of Ergonomic Office Chair (SKU: CHR-204) at Rs 8,500 each
- 20 units of Standing Desk (SKU: DSK-118) at Rs 22,000 each
- 100 units of USB-C Hub (SKU: HUB-77) at Rs 1,200 each

Payment terms: Net 30
Requested delivery by: 31 March 2025
Contact: procurement@greenleaf.in
"""

schema = (
    "po_number (string), "
    "order_date (ISO-8601 date), "
    "ship_to_address (string), "
    "line_items (array of objects, each with: sku (string), description (string), quantity (integer), unit_price_inr (number)), "
    "total_amount_inr (number), "
    "payment_terms (string), "
    "delivery_deadline (ISO-8601 date), "
    "contact_email (string)"
)

result = extract(purchase_order, schema)
print(json.dumps(result, indent=2))

**Expected output:**

```json
{
  "po_number": "PO-9921",
  "order_date": "2025-03-20",
  "ship_to_address": "45 Innovation Drive, Bengaluru 560001",
  "line_items": [
    {"sku": "CHR-204", "description": "Ergonomic Office Chair", "quantity": 50, "unit_price_inr": 8500},
    {"sku": "DSK-118", "description": "Standing Desk",          "quantity": 20, "unit_price_inr": 22000},
    {"sku": "HUB-77",  "description": "USB-C Hub",              "quantity": 100,"unit_price_inr": 1200}
  ],
  "total_amount_inr": 994000,
  "payment_terms": "Net 30",
  "delivery_deadline": "2025-03-31",
  "contact_email": "procurement@greenleaf.in"
}
```

### Step 9 — Batch Processing Multiple Documents

In production you often need to process hundreds or thousands of documents. The cell below shows how to batch-process a list of texts with the same schema and collect results into a list.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def batch_extract(
    documents: list[str],
    schema_description: str,
    max_workers: int = 4,
) -> list[dict | None]:
    """Extract structured fields from a list of documents concurrently.

    Args:
        documents: List of unstructured text strings.
        schema_description: Plain-English field descriptions (same for all docs).
        max_workers: Number of parallel threads (keep low to respect API rate limits).

    Returns:
        List of result dicts in the same order as `documents`.
        Failed extractions produce None in that position.
    """
    results   = [None] * len(documents)
    futures   = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for idx, doc in enumerate(documents):
            future = executor.submit(extract, doc, schema_description)
            futures[future] = idx

        for future in as_completed(futures):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as exc:
                print(f"  Document {idx} failed: {exc}")

    return results


# --- Demo: run all three sample documents through the same schema ---
contact_schema = "person_name (string), email (string or null), organisation (string or null)"

sample_docs = [
    "Hi, I'm Priya Sharma from TechNova Solutions. Reach me at priya@technova.io.",
    "Meeting request from Daniel Osei (d.osei@globalbank.com), VP of Engineering at Global Bank.",
    "No contact info here — just a general announcement about the office closure next Monday.",
]

batch_results = batch_extract(sample_docs, contact_schema)
for doc, res in zip(sample_docs, batch_results):
    print(f"Input : {doc[:60]}...")
    print(f"Output: {json.dumps(res, indent=2)}\n")

**Expected output:**

```
Input : Hi, I'm Priya Sharma from TechNova Solutions. Reach me ...
Output: {
  "person_name": "Priya Sharma",
  "email": "priya@technova.io",
  "organisation": "TechNova Solutions"
}

Input : Meeting request from Daniel Osei (d.osei@globalbank.com)...
Output: {
  "person_name": "Daniel Osei",
  "email": "d.osei@globalbank.com",
  "organisation": "Global Bank"
}

Input : No contact info here — just a general announcement abou...
Output: {
  "person_name": null,
  "email": null,
  "organisation": null
}
```

Notice that the third document returns `null` for all fields — the model correctly identifies when information is absent rather than hallucinating a value.

### Step 10 — Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `EnvironmentError: NUGEN_API_KEY not found` | `.env` missing or not in notebook directory | Create `.env` with `NUGEN_API_KEY=your_key` |
| `RuntimeError: Nugen API unreachable` | Invalid key or network issue | Verify key at [Nugen Dashboard](https://nugen-platform-frontend.azurewebsites.net/dashboard) |
| `ValueError: Model returned non-JSON output` | Model added extra prose | Lower `temperature` (already 0.0); refine schema description to be more explicit |
| Field value is `null` when it should not be | Field wording unclear | Make field name more descriptive, e.g. `order_id` → `order_number from the subject line (string)` |
| Wrong enum value returned | Enum options not covering the case | Add more enum options or use `string` instead of a fixed enum |
| Slow for large batches | API rate limits | Reduce `max_workers` in `batch_extract()` or add `time.sleep()` between batches |

### Useful Links
- [Nugen API Reference](https://docs.nugen.in/api-reference)
- [Nugen Chat Completions endpoint](https://docs.nugen.in/api-reference/chat-completions)
- [Nugen model overview](https://docs.nugen.in/introduction)
- [Nugen Dashboard](https://nugen-platform-frontend.azurewebsites.net/dashboard)

### Conclusion

You have built a general-purpose structured information extraction pipeline with Nugen:

| What | How |
|---|---|
| **Any unstructured text** | Email, incident report, purchase order, contract, form |
| **Any target schema** | Described in plain English — no JSON Schema, no training data |
| **Reliable output** | Temperature 0.0 + code-fence stripping + JSON validation |
| **Production-ready** | Retry logic, batch processing, clear error messages |

**Ideas for extending this notebook:**
- Add a JSON Schema validator (e.g. `jsonschema` library) to enforce field types after extraction
- Stream results to a database (PostgreSQL, BigQuery) using `psycopg2` or `google-cloud-bigquery`
- Build a simple FastAPI endpoint that wraps `extract()` for use as a microservice
- Swap `nugen-flash-instruct` for a domain-specific Nugen model (e.g. legal, medical) for higher accuracy on regulated documents